# Assignment 2

In this assigment, we will work with the *Forest Fire* data set. Please download the data from the [UCI Machine Learning Repository](https://archive.ics.uci.edu/dataset/162/forest+fires). Extract the data files into the subdirectory: `../data/fires/` (relative to `./05_src/`).

## Objective

+ The model objective is to predict the area affected by forest fires given the features set. 
+ The objective of this exercise is to assess your ability to construct and evaluate model pipelines.
+ Please note: the instructions are not meant to be 100% prescriptive, but instead they are a set of minimum requirements. If you find predictive performance gains by applying additional steps, by all means show them. 

## Variable Description

From the description file contained in the archive (`forestfires.names`), we obtain the following variable descriptions:

1. X - x-axis spatial coordinate within the Montesinho park map: 1 to 9
2. Y - y-axis spatial coordinate within the Montesinho park map: 2 to 9
3. month - month of the year: "jan" to "dec" 
4. day - day of the week: "mon" to "sun"
5. FFMC - FFMC index from the FWI system: 18.7 to 96.20
6. DMC - DMC index from the FWI system: 1.1 to 291.3 
7. DC - DC index from the FWI system: 7.9 to 860.6 
8. ISI - ISI index from the FWI system: 0.0 to 56.10
9. temp - temperature in Celsius degrees: 2.2 to 33.30
10. RH - relative humidity in %: 15.0 to 100
11. wind - wind speed in km/h: 0.40 to 9.40 
12. rain - outside rain in mm/m2 : 0.0 to 6.4 
13. area - the burned area of the forest (in ha): 0.00 to 1090.84 









### Specific Tasks

+ Construct four model pipelines, out of combinations of the following components:

    + Preprocessors:

        - A simple processor that only scales numeric variables and recodes categorical variables.
        - A transformation preprocessor that scales numeric variables and applies a non-linear transformation.
    
    + Regressor:

        - A baseline regressor, which could be a [K-nearest neighbours model]() or a linear model like [Lasso](https://scikit-learn.org/stable/modules/generated/sklearn.linear_model.Lasso.html) or [Ridge Regressors](https://scikit-learn.org/stable/modules/generated/sklearn.linear_model.ridge_regression.html).
        - An advanced regressor of your choice (e.g., Bagging, Boosting, SVR, etc.). TIP: select a tree-based method such that it does not take too long to run SHAP further below. 

+ Evaluate tune and evaluate each of the four model pipelines. 

    - Select a [performance metric](https://scikit-learn.org/stable/modules/linear_model.html) out of the following options: explained variance, max error, root mean squared error (RMSE), mean absolute error (MAE), r-squared.
    - *TIPS*: 
    
        * Out of the suggested metrics above, [some are correlation metrics, but this is a prediction problem](https://www.tmwr.org/performance#performance). Choose wisely (and don't choose the incorrect options.) 

+ Select the best-performing model and explain its predictions.

    - Provide local explanations.
    - Obtain global explanations and recommend a variable selection strategy.

+ Export your model as a pickle file.


You can work on the Jupyter notebook, as this experiment is fairly short (no need to use sacred). 

# Load the data

Place the files in the ../../05_src/data/fires/ directory and load the appropriate file. 

In [5]:
# Load the libraries as required.

import pandas as pd
import numpy as np

from sklearn.preprocessing import OneHotEncoder, StandardScaler, FunctionTransformer
from sklearn.compose import ColumnTransformer

from sklearn.pipeline import Pipeline

from sklearn.tree import DecisionTreeRegressor
from sklearn.model_selection import GridSearchCV
from sklearn.ensemble import RandomForestRegressor
import matplotlib.pyplot as plt

In [ ]:
# Load data
columns = [
    'coord_x', 'coord_y', 'month', 'day', 'ffmc', 'dmc', 'dc', 'isi', 'temp', 'rh', 'wind', 'rain', 'area' 
]
fires_dt = (pd.read_csv('../../05_src/data/fires/forestfires.csv', header = 0, names = columns))
fires_dt.info()


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 517 entries, 0 to 516
Data columns (total 13 columns):
 #   Column   Non-Null Count  Dtype  
---  ------   --------------  -----  
 0   coord_x  517 non-null    int64  
 1   coord_y  517 non-null    int64  
 2   month    517 non-null    object 
 3   day      517 non-null    object 
 4   ffmc     517 non-null    float64
 5   dmc      517 non-null    float64
 6   dc       517 non-null    float64
 7   isi      517 non-null    float64
 8   temp     517 non-null    float64
 9   rh       517 non-null    int64  
 10  wind     517 non-null    float64
 11  rain     517 non-null    float64
 12  area     517 non-null    float64
dtypes: float64(8), int64(3), object(2)
memory usage: 52.6+ KB


# Get X and Y

Create the features data frame and target data.

In [7]:
# Target variable
y = fires_dt['area']

# Feature variables
X = fires_dt.drop('area', axis=1)


# Preprocessing

Create two [Column Transformers](https://scikit-learn.org/stable/modules/generated/sklearn.compose.ColumnTransformer.html), called preproc1 and preproc2, with the following guidelines:

- Numerical variables

    * (Preproc 1 and 2) Scaling: use a scaling method of your choice (Standard, Robust, Min-Max). 
    * Preproc 2 only: 
        
        + Choose a transformation for any of your input variables (or several of them). Evaluate if this transformation is convenient.
        + The choice of scaler is up to you.

- Categorical variables: 
    
    * (Preproc 1 and 2) Apply [one-hot encoding](https://scikit-learn.org/stable/modules/generated/sklearn.preprocessing.OneHotEncoder.html) where appropriate.


+ The only difference between preproc1 and preproc2 is the non-linear transformation of the numerical variables.
    


### Preproc 1

Create preproc1 below.

+ Numeric: scaled variables, no other transforms.
+ Categorical: one-hot encoding.

In [ ]:
# categorical and numerical columns
categorical_cols = ['month', 'day']
numerical_cols = [col for col in X.columns if col not in categorical_cols]

# Preproc1
preproc1 = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), numerical_cols),
         ('cat', OneHotEncoder(drop='first', handle_unknown='ignore'), categorical_cols)
    ]
)

### Preproc 2

Create preproc1 below.

+ Numeric: scaled variables, non-linear transformation to one or more variables.
+ Categorical: one-hot encoding.

In [ ]:
#categorical and numerical columns
categorical_cols = ['month', 'day']
numerical_cols = [col for col in X.columns if col not in categorical_cols]
transformed_number = ['dc']
remaining_number = [col for col in numerical_cols if col not in transformed_number]

#non-linear transformer
transformer = FunctionTransformer(np.log1p)
scaler = StandardScaler()
one_hot_encoder = OneHotEncoder(drop='first', handle_unknown='ignore',)

# preproc2
preproc2 = ColumnTransformer(
    transformers=[
        ('dc', transformer, transformed_number),   
        ('scale_num', scaler, remaining_number),        
        ('encode_cat', one_hot_encoder, categorical_cols) 
    ]
)

## Model Pipeline


Create a [model pipeline](https://scikit-learn.org/stable/modules/generated/sklearn.pipeline.Pipeline.html): 

+ Add a step labelled `preprocessing` and assign the Column Transformer from the previous section.
+ Add a step labelled `regressor` and assign a regression model to it. 

## Regressor

+ Use a regression model to perform a prediction. 

    - Choose a baseline regressor, tune it (if necessary) using grid search, and evaluate it using cross-validation.
    - Choose a more advance regressor, tune it (if necessary) using grid search, and evaluate it using cross-validation.
    - Both model choices are up to you, feel free to experiment.

In [10]:
# Pipeline A = preproc1 + baseline

pipeline_A = Pipeline(steps=[
    ('preprocessing', preproc1),   
    ('regressor', DecisionTreeRegressor(random_state=42))
])

In [11]:
# Pipeline B = preproc2 + baseline

pipeline_B = Pipeline(steps=[
    ('preprocessing', preproc2),  
    ('regressor', DecisionTreeRegressor(random_state=42)) 
])


In [12]:
# Pipeline C = preproc1 + advanced model

pipeline_C = Pipeline(steps=[
    ('preprocessing', preproc1), 
    ('regressor', RandomForestRegressor(random_state=42)) 
])

In [13]:
# Pipeline D = preproc2 + advanced model

    
pipeline_D = Pipeline(steps=[
    ('preprocessing', preproc2), 
    ('regressor', DecisionTreeRegressor(random_state=42))
])

# Tune Hyperparams

+ Perform GridSearch on each of the four pipelines. 
+ Tune at least one hyperparameter per pipeline.
+ Experiment with at least four value combinations per pipeline.

In [14]:
# Pipeline A = preproc1 + baseline
pipeline_A = Pipeline(steps=[
    ('preprocessing', preproc1),   
    ('regressor', DecisionTreeRegressor(random_state=42))
])


param_grid = {
    'regressor__max_depth': [5, 10, 20],
    'regressor__min_samples_split': [2, 5]
}

# Grid search using R squared
grid = GridSearchCV(pipeline_A, param_grid, cv=5, scoring='r2')

# Fit
grid.fit(X, y)

# Results
print("Best Params:", grid.best_params_)
print("Best R² Score:", grid.best_score_)

/Applications/miniconda3/envs/dsi_participant/lib/python3.9/site-packages/sklearn/preprocessing/_encoders.py:202: UserWarning: Found unknown categories in columns [0] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(
/Applications/miniconda3/envs/dsi_participant/lib/python3.9/site-packages/sklearn/preprocessing/_encoders.py:202: UserWarning: Found unknown categories in columns [0] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(
/Applications/miniconda3/envs/dsi_participant/lib/python3.9/site-packages/sklearn/preprocessing/_encoders.py:202: UserWarning: Found unknown categories in columns [0] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(
/Applications/miniconda3/envs/dsi_participant/lib/python3.9/site-packages/sklearn/preprocessing/_encoders.py:202: UserWarning: Found unknown categories in columns [0] during transform. These unknown categories will be encoded as all

Best Params: {'regressor__max_depth': 5, 'regressor__min_samples_split': 5}
Best R² Score: -5.25900713206127


In [ ]:
# Pipeline B = preproc2 + baseline

pipeline_B = Pipeline(steps=[
    ('preprocessing', preproc2),  
    ('regressor', DecisionTreeRegressor(random_state=42)) 
])

param_grid = {
    'regressor__max_depth': [20, 30, 40],
    'regressor__min_samples_split': [5, 10]
}

# 
grid_B = GridSearchCV(
    pipeline_B, 
    param_grid, 
    cv=5, 
    scoring='r2', 
)

# Fit
grid_B.fit(X, y)

print("Best Params (preproc2):", grid_B.best_params_)
print("Best R² Score (preproc2):", grid_B.best_score_)

/Applications/miniconda3/envs/dsi_participant/lib/python3.9/site-packages/sklearn/preprocessing/_encoders.py:202: UserWarning: Found unknown categories in columns [0] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(
/Applications/miniconda3/envs/dsi_participant/lib/python3.9/site-packages/sklearn/preprocessing/_encoders.py:202: UserWarning: Found unknown categories in columns [0] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(
/Applications/miniconda3/envs/dsi_participant/lib/python3.9/site-packages/sklearn/preprocessing/_encoders.py:202: UserWarning: Found unknown categories in columns [0] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(
/Applications/miniconda3/envs/dsi_participant/lib/python3.9/site-packages/sklearn/preprocessing/_encoders.py:202: UserWarning: Found unknown categories in columns [0] during transform. These unknown categories will be encoded as all

Best Params (preproc2): {'regressor__max_depth': 20, 'regressor__min_samples_split': 10}
Best R² Score (preproc2): -8.023055653998806


/Applications/miniconda3/envs/dsi_participant/lib/python3.9/site-packages/sklearn/preprocessing/_encoders.py:202: UserWarning: Found unknown categories in columns [0] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(
/Applications/miniconda3/envs/dsi_participant/lib/python3.9/site-packages/sklearn/preprocessing/_encoders.py:202: UserWarning: Found unknown categories in columns [0] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(
/Applications/miniconda3/envs/dsi_participant/lib/python3.9/site-packages/sklearn/preprocessing/_encoders.py:202: UserWarning: Found unknown categories in columns [0] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(
/Applications/miniconda3/envs/dsi_participant/lib/python3.9/site-packages/sklearn/preprocessing/_encoders.py:202: UserWarning: Found unknown categories in columns [0] during transform. These unknown categories will be encoded as all

In [16]:
# Pipeline C = preproc1 + advanced model

pipeline_C = Pipeline(steps=[
    ('preprocessing', preproc1), 
    ('regressor', RandomForestRegressor(random_state=42)) 
])



param_grid = {
    'regressor__max_depth': [5, 10, 20],
    'regressor__min_samples_split': [2, 5]
}

# Grid search using R²
grid_C = GridSearchCV(pipeline_C, param_grid, cv=5, scoring='r2')

# Fit
grid_C.fit(X, y)

# # Results
print("Best Params:", grid.best_params_)
print("Best R² Score:", grid.best_score_)

/Applications/miniconda3/envs/dsi_participant/lib/python3.9/site-packages/sklearn/preprocessing/_encoders.py:202: UserWarning: Found unknown categories in columns [0] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(
/Applications/miniconda3/envs/dsi_participant/lib/python3.9/site-packages/sklearn/preprocessing/_encoders.py:202: UserWarning: Found unknown categories in columns [0] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(
/Applications/miniconda3/envs/dsi_participant/lib/python3.9/site-packages/sklearn/preprocessing/_encoders.py:202: UserWarning: Found unknown categories in columns [0] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(
/Applications/miniconda3/envs/dsi_participant/lib/python3.9/site-packages/sklearn/preprocessing/_encoders.py:202: UserWarning: Found unknown categories in columns [0] during transform. These unknown categories will be encoded as all

Best Params: {'regressor__max_depth': 5, 'regressor__min_samples_split': 5}
Best R² Score: -5.25900713206127


In [17]:
# Pipeline D = preproc2 + advanced model

    
pipeline_D = Pipeline(steps=[
    ('preprocessing', preproc2), 
    ('regressor', DecisionTreeRegressor(random_state=42))
])


param_grid = {
    'regressor__max_depth': [5, 10, 20],      
    'regressor__min_samples_split': [2, 5]
}

# Grid search using R²
grid_D = GridSearchCV(pipeline_D, param_grid, cv=5, scoring='r2')

# Fit
grid_D.fit(X, y)

# Results
print("Best Params:", grid.best_params_)
print("Best R² Score:", grid.best_score_)

/Applications/miniconda3/envs/dsi_participant/lib/python3.9/site-packages/sklearn/preprocessing/_encoders.py:202: UserWarning: Found unknown categories in columns [0] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(
/Applications/miniconda3/envs/dsi_participant/lib/python3.9/site-packages/sklearn/preprocessing/_encoders.py:202: UserWarning: Found unknown categories in columns [0] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(
/Applications/miniconda3/envs/dsi_participant/lib/python3.9/site-packages/sklearn/preprocessing/_encoders.py:202: UserWarning: Found unknown categories in columns [0] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(
/Applications/miniconda3/envs/dsi_participant/lib/python3.9/site-packages/sklearn/preprocessing/_encoders.py:202: UserWarning: Found unknown categories in columns [0] during transform. These unknown categories will be encoded as all

Best Params: {'regressor__max_depth': 5, 'regressor__min_samples_split': 5}
Best R² Score: -5.25900713206127


/Applications/miniconda3/envs/dsi_participant/lib/python3.9/site-packages/sklearn/preprocessing/_encoders.py:202: UserWarning: Found unknown categories in columns [0] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(
/Applications/miniconda3/envs/dsi_participant/lib/python3.9/site-packages/sklearn/preprocessing/_encoders.py:202: UserWarning: Found unknown categories in columns [0] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(
/Applications/miniconda3/envs/dsi_participant/lib/python3.9/site-packages/sklearn/preprocessing/_encoders.py:202: UserWarning: Found unknown categories in columns [0] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(
/Applications/miniconda3/envs/dsi_participant/lib/python3.9/site-packages/sklearn/preprocessing/_encoders.py:202: UserWarning: Found unknown categories in columns [0] during transform. These unknown categories will be encoded as all

# Evaluate

+ Which model has the best performance?

Model C has the best performance using a decision tree regressor.

In [18]:
print("A:", grid.best_params_, "R2:", grid.best_score_)
print("B:", grid_B.best_params_, "R2:", grid_B.best_score_)
print("C:", grid_C.best_params_, "R2:", grid_C.best_score_)
print("D:", grid_D.best_params_, "R2:", grid_D.best_score_)

A: {'regressor__max_depth': 5, 'regressor__min_samples_split': 5} R2: -5.25900713206127
B: {'regressor__max_depth': 20, 'regressor__min_samples_split': 10} R2: -8.023055653998806
C: {'regressor__max_depth': 5, 'regressor__min_samples_split': 5} R2: -4.101532360859613
D: {'regressor__max_depth': 5, 'regressor__min_samples_split': 5} R2: -4.118534023271744


# Export

+ Save the best performing model to a pickle file.

In [20]:
import pickle

# Save model B
with open('model_C.pkl', 'wb') as file:
    pickle.dump(grid_C.best_estimator_, file)

# Explain

+ Use SHAP values to explain the following only for the best-performing model:

    - Select an observation in your test set and explain which are the most important features that explain that observation's specific prediction.

    - In general, across the complete training set, which features are the most and least important.

+ If you were to remove features from the model, which ones would you remove? Why? How would you test that these features are actually enhancing model performance?

In [ ]:
import shap


data_transform = pipeline_C.named_steps['preprocessing'].transform(X)

explainer = shap.TreeExplainer(
    pipeline_C.named_steps['regressor'], 
    data_transform,
    feature_names = pipeline_C.named_steps['preprocessing'].get_feature_names_out())

shap_values = explainer.shap_values(data_transform)

shap.summary_plot(shap_values, X_transformed, feature_names=feature_names)


#I was having issues with importing shap. I have tried many different options install and uninstall different softwares to ensure my environment is compatible for using shap.
#I was not able to resolve the issue. 
#this code is my attempt at plotting the shap values. 
#the features that are the most important would be the ones that have the most contribution to the prediction. least important features would be the features that have no contribution to the prediction.
#If I was to remove a feature from the model, I would remove the feature with the least contribution to the prediction based on the shap values. 
# I would then test the model with and without the features to determine if the expectations turn out to be true. 


SystemError: initialization of _internal failed without raising an exception

*(Answer here.)*

## Criteria

The [rubric](./assignment_2_rubric_clean.xlsx) contains the criteria for assessment.

## Submission Information

🚨 **Please review our [Assignment Submission Guide](https://github.com/UofT-DSI/onboarding/blob/main/onboarding_documents/submissions.md)** 🚨 for detailed instructions on how to format, branch, and submit your work. Following these guidelines is crucial for your submissions to be evaluated correctly.

### Submission Parameters:
* Submission Due Date: `HH:MM AM/PM - DD/MM/YYYY`
* The branch name for your repo should be: `assignment-2`
* What to submit for this assignment:
    * This Jupyter Notebook (assignment_2.ipynb) should be populated and should be the only change in your pull request.
* What the pull request link should look like for this assignment: `https://github.com/<your_github_username>/production/pull/<pr_id>`
    * Open a private window in your browser. Copy and paste the link to your pull request into the address bar. Make sure you can see your pull request properly. This helps the technical facilitator and learning support staff review your submission easily.

Checklist:
- [ ] Created a branch with the correct naming convention.
- [ ] Ensured that the repository is public.
- [ ] Reviewed the PR description guidelines and adhered to them.
- [ ] Verify that the link is accessible in a private browser window.

If you encounter any difficulties or have questions, please don't hesitate to reach out to our team via our Slack at the `help` channel. Our Technical Facilitators and Learning Support staff are here to help you navigate any challenges.

# Reference

Cortez,Paulo and Morais,Anbal. (2008). Forest Fires. UCI Machine Learning Repository. https://doi.org/10.24432/C5D88D.